In [2]:
import pandas as pd
patients=pd.read_csv("patients_clean.csv")
services_weekly=pd.read_csv("services_weekly.csv")
staff=pd.read_csv("staff_clean.csv")
staff_schedule=pd.read_csv("staff_schedule_clean.csv")

In [3]:
total_admissions = (
    patients
    .groupby("service")
    .size()
    .reset_index(name="total_admissions")
)
total_admissions

,service,total_admissions
0,Emergency,263
1,General_Medicine,242
2,Icu,241
3,Surgery,254


In [5]:
avg_los = (
    patients
    .groupby("service")["length_of_stay"]
    .mean()
    .reset_index(name="average_length_of_stay")
)
avg_los

,service,average_length_of_stay
0,Emergency,7.159696
1,General_Medicine,6.995868
2,Icu,7.605809
3,Surgery,7.866142


In [15]:
hospital_readmission_dataset["readmitted_flag"] = (
    hospital_readmission_dataset["readmitted"]
    .str.lower()
    .eq("yes")
    .astype(int)
)

readmission_rate = (
    hospital_readmission_dataset
    .groupby("service")
    .agg(
        readmission_rate=("readmitted_flag", "mean")
    )
    .reset_index()
)

readmission_rate["readmission_rate"] *= 100
readmission_rate["readmission_rate"] = (
    readmission_rate["readmission_rate"].round(2)
)

print(readmission_rate)

            service  readmission_rate
0         Emergency             21.67
1  General_Medicine             16.12
2               Icu             19.50
3           Surgery             20.08


In [8]:

department_summary = (
    services_weekly
    .groupby("service")
    .agg({
        "patients_admitted": "sum",
        "patients_refused": "sum",
        "patient_satisfaction": "mean",
        "staff_morale": "mean"
    })
    .reset_index()
)


department_summary["admission_rate"] = (
    department_summary["patients_admitted"] /
    (
        department_summary["patients_admitted"] +
        department_summary["patients_refused"]
    )
) * 100
department_summary["department_efficiency_score"] = (
      0.40 * department_summary["admission_rate"]
    + 0.30 * department_summary["patient_satisfaction"]
    + 0.30 * department_summary["staff_morale"]
)

department_summary["admission_rate"] = (
    department_summary["admission_rate"].round(2)
)

department_summary["patient_satisfaction"] = (
    department_summary["patient_satisfaction"].round(2)
)

department_summary["staff_morale"] = (
    department_summary["staff_morale"].round(2)
)

department_summary["department_efficiency_score"] = (
    department_summary["department_efficiency_score"].round(2)
)

print("\n========== Department Efficiency ==========\n")
print(department_summary)

department_summary.to_excel(
    "department_efficiency_score.xlsx",
    index=False
)

print("\nDepartment Efficiency Score saved successfully!")


========== Department Efficiency ==========

            service  patients_admitted  patients_refused  \
0               ICU                648               141   
1         emergency               1185              5008   
2  general_medicine               2332              1938   
3           surgery               1686               555   

   patient_satisfaction  staff_morale  admission_rate  \
0                 81.62         70.98           82.13   
1                 77.88         73.56           19.13   
2                 81.23         73.10           54.61   
3                 79.27         72.63           75.23   

   department_efficiency_score  
0                        78.63  
1                        53.09  
2                        68.14  
3                        75.66  

Department Efficiency Score saved successfully!


In [9]:

occupancy = (
    services_weekly
    .groupby("service")
    .agg({
        "patients_admitted": "sum",
        "available_beds": "sum"
    })
    .reset_index()
)

occupancy["occupancy_rate"] = (
    occupancy["patients_admitted"] /
    occupancy["available_beds"]
) * 100

occupancy["occupancy_rate"] = occupancy["occupancy_rate"].round(2)


print(occupancy)

occupancy.to_excel(
    "occupancy_rate.xlsx",
    index=False
)

            service  patients_admitted  available_beds  occupancy_rate
0               ICU                648             772           83.94
1         emergency               1185            1185          100.00
2  general_medicine               2332            2404           97.00
3           surgery               1686            1951           86.42


In [10]:

bed_utilization = (
    services_weekly
    .groupby("service")
    .agg({
        "patients_admitted": "sum",
        "available_beds": "sum"
    })
    .reset_index()
)
bed_utilization["bed_utilization_rate"] = (
    bed_utilization["patients_admitted"] /
    (
        bed_utilization["patients_admitted"] +
        bed_utilization["available_beds"]
    )
) * 100

bed_utilization["bed_utilization_rate"] = (
    bed_utilization["bed_utilization_rate"].round(2)
)
print(bed_utilization)

bed_utilization.to_excel(
    "bed_utilization_rate.xlsx",
    index=False
)

            service  patients_admitted  available_beds  bed_utilization_rate
0               ICU                648             772                 45.63
1         emergency               1185            1185                 50.00
2  general_medicine               2332            2404                 49.24
3           surgery               1686            1951                 46.36


In [20]:
occupancy = occupancy[
    ["service", "occupancy_rate"]
]

bed_utilization = bed_utilization[
    ["service", "bed_utilization_rate"]
]

department_efficiency = department_efficiency[
    ["service", "department_efficiency_score"]
]

In [21]:
step1 = total_admissions.merge(occupancy, on="service", how="left")
print("Step 1 OK")

step2 = step1.merge(bed_utilization, on="service", how="left")
print("Step 2 OK")

step3 = step2.merge(avg_los, on="service", how="left")
print("Step 3 OK")

step4 = step3.merge(readmission_rate, on="service", how="left")
print("Step 4 OK")

step5 = step4.merge(department_efficiency, on="service", how="left")
print("Step 5 OK")

Step 1 OK
Step 2 OK
Step 3 OK
Step 4 OK
Step 5 OK


In [22]:
hospital_final_dataset = step5

hospital_final_dataset.to_csv(
    "hospital_final_dataset.csv",
    index=False
)

print(hospital_final_dataset)

            service  total_admissions  occupancy_rate  bed_utilization_rate  \
0         Emergency               263             NaN                   NaN   
1  General_Medicine               242             NaN                   NaN   
2               Icu               241             NaN                   NaN   
3           Surgery               254             NaN                   NaN   

   average_length_of_stay  readmission_rate  department_efficiency_score  
0                7.159696             21.67                          NaN  
1                6.995868             16.12                          NaN  
2                7.605809             19.50                          NaN  
3                7.866142             20.08                          NaN  


In [23]:
print(total_admissions["service"])

print(occupancy["service"])

print(bed_utilization["service"])

print(avg_los["service"])

print(readmission_rate["service"])

print(department_efficiency["service"])

0           Emergency
1    General_Medicine
2                 Icu
3             Surgery
Name: service, dtype: object
0                 ICU
1           emergency
2    general_medicine
3             surgery
Name: service, dtype: object
0                 ICU
1           emergency
2    general_medicine
3             surgery
Name: service, dtype: object
0           Emergency
1    General_Medicine
2                 Icu
3             Surgery
Name: service, dtype: object
0           Emergency
1    General_Medicine
2                 Icu
3             Surgery
Name: service, dtype: object
0                 ICU
1           emergency
2    general_medicine
3             surgery
Name: service, dtype: object


In [24]:
# List of all DataFrames
dfs = [
    total_admissions,
    occupancy,
    bed_utilization,
    avg_los,
    readmission_rate,
    department_efficiency
]

# Standardize service names
for df in dfs:
    df["service"] = (
        df["service"]
        .str.strip()          # Remove leading/trailing spaces
        .str.lower()          # Convert to lowercase
    )

In [25]:
hospital_final_dataset = (
    total_admissions
    .merge(occupancy, on="service", how="left")
    .merge(bed_utilization, on="service", how="left")
    .merge(avg_los, on="service", how="left")
    .merge(readmission_rate, on="service", how="left")
    .merge(department_efficiency, on="service", how="left")
)

In [26]:
hospital_final_dataset["service"] = (
    hospital_final_dataset["service"]
    .str.replace("_", " ")
    .str.title()
)

In [28]:
hospital_final_dataset.head()

,service,total_admissions,occupancy_rate,bed_utilization_rate,average_length_of_stay,readmission_rate,department_efficiency_score
0,Emergency,263,100.00,50.00,7.159696,21.67,53.09
1,General Medicine,242,97.00,49.24,6.995868,16.12,68.14
2,Icu,241,83.94,45.63,7.605809,19.50,78.63
3,Surgery,254,86.42,46.36,7.866142,20.08,75.66


In [29]:
import pandas as pd


# Overall KPI Calculations



overall_total_admissions = len(patients)


overall_occupancy_rate = (
    services_weekly["patients_admitted"].sum()
    / services_weekly["available_beds"].sum()
) * 100


overall_bed_utilization_rate = (
    services_weekly["patients_admitted"].sum()
    /
    (
        services_weekly["patients_admitted"].sum()
        + services_weekly["available_beds"].sum()
    )
) * 100


overall_average_los = patients["length_of_stay"].mean()


overall_readmission_rate = (
    (
        hospital_readmission_dataset["readmitted"]
        .str.lower()
        .eq("yes")
        .mean()
    )
    * 100
)


overall_department_efficiency = (
    department_efficiency["department_efficiency_score"].mean()
)



overall_hospital_kpis = pd.DataFrame({
    "total_admissions": [overall_total_admissions],
    "occupancy_rate": [round(overall_occupancy_rate, 2)],
    "bed_utilization_rate": [round(overall_bed_utilization_rate, 2)],
    "average_length_of_stay": [round(overall_average_los, 2)],
    "readmission_rate": [round(overall_readmission_rate, 2)],
    "department_efficiency_score": [round(overall_department_efficiency, 2)]
})



overall_hospital_kpis.to_csv(
    "overall_hospital_kpis.csv",
    index=False
)

print("\nOverall Hospital KPIs\n")
print(overall_hospital_kpis)


Overall Hospital KPIs

   total_admissions  occupancy_rate  bed_utilization_rate  \
0              1000            92.7                  48.1   

   average_length_of_stay  readmission_rate  department_efficiency_score  
0                    7.41              19.4                        68.88  
